# Expectations about inflation, house prices, and the stock market

TECH2 term paper: an analysis of US household expectations using the Survey of Consumer Expectations (SCE), run by the Federal Reserve Bank of New York.

## Setup

In [1]:
import glob

import pandas as pd

## Part 1: Reading in the data

Each survey wave is stored in its own file named `SCE-MMM-YYYY.csv`. Each row is one respondent in that month.

### 1.1 Read and merge the SCE files

The files are semicolon-separated, so `sep=';'` is required. The pattern `SCE-*.csv` leaves out `CPI.csv`, which is in the same folder.

Filenames don't sort chronologically (`Apr-2014` comes before `Jun-2013`), so the merged data is sorted by date. Part 2 relies on this order to find each respondent's first observation.

In [2]:
files = glob.glob('data/SCE-*.csv')

# Read each survey wave and stack them into a single DataFrame
df_raw = pd.concat(
    [pd.read_csv(file, sep=';', parse_dates=['date']) for file in files],
    ignore_index=True,
)

# Put the observations in chronological order
df_raw = df_raw.sort_values(['date', 'userid'], ignore_index=True)

print(f'Read {len(files)} files into {df_raw.shape[1]} columns')

Read 139 files into 29 columns


### 1.2 Sample statistics

These statistics are reported again for the cleaned data in Part 2, so they are computed by a reusable function.

In [3]:
def sample_statistics(df):
    """
    Summarize the size and time span of a survey data set.

    Parameters
    ----------
    df : pandas.DataFrame
        Survey data with the columns ``userid``, ``wid`` and ``date``.

    Returns
    -------
    pandas.DataFrame
        Table with one row per statistic.
    """
    statistics = {
        'Unique individuals': f'{df["userid"].nunique():,}',
        'Observations': f'{len(df):,}',
        'Unique survey waves': f'{df["wid"].nunique():,}',
        'First date': f'{df["date"].min():%Y-%m-%d}',
        'Last date': f'{df["date"].max():%Y-%m-%d}',
    }

    return pd.DataFrame.from_dict(statistics, orient='index', columns=['Value'])

In [4]:
sample_statistics(df_raw)

,Value
Unique individuals,"23,369"
Observations,"176,101"
Unique survey waves,139
First date,2013-06-01
Last date,2024-12-31


The raw data has 176,101 observations from 23,369 individuals across 139 monthly waves, or about 7.5 observations per individual. That is fewer than the usual 12 because some respondents stop early, and those at the end of the sample haven't finished their 12 months.

The data starts in June 2013, before the 2014 start given in the task description.